# Ingesta RRS 2025–2028: reglas → CSV

Se lee `corpus/2025-2028-RRS-with-Changes-and-Corrections.pdf` con **pypdf** (mismo enfoque que `regatas_assistant/ingestion.py`).

**Ejecutá las celdas en orden** (importaciones → lectura PDF → limpieza → **definiciones** → regex de reglas → tabla → CSV).

Se detectan **reglas numeradas** y de **apéndice** con expresiones regulares sobre el texto extraído:

- **Partes 1–7:** encabezados `N TÍTULO` (una línea, solo espacios/tabs entre número y título) y subreglas `N.M` / `N.M.P` al inicio de línea.
- **Desde Appendix A:** mismas subreglas numéricas y reglas tipo `A2.1`, `C3.1`, etc.
- **Definiciones:** bloque *DEFINITIONS* … *BASIC PRINCIPLES* (celda aparte) → **`scripts/definitions.csv`** (`id`, `termino`, `texto`).

Salida de reglas: **`scripts/rrs_reglas_2025_2028.csv`** (`numero_regla`, `titulo`, `tipo`, `seccion`, `texto`). **titulo** es el encabezado de la regla sin el identificador (p. ej. `10` → `ON OPPOSITE TACKS`). La columna **seccion** toma la última cabecera **SECTION …** o **Appendix …** (título en la línea siguiente) anterior a la regla en el texto extraído.

**Nota:** el PDF a veces une o parte líneas; la tabla es una aproximación útil para EDA.


## 1. Importaciones y constantes


In [40]:
from __future__ import annotations

import csv
import os
import re
from pathlib import Path

from pypdf import PdfReader

PDF_NAME = "2025-2028-RRS-with-Changes-and-Corrections.pdf"
OUT_NAME = "rrs_reglas_2025_2028.csv"
DEFINITIONS_OUT = "definitions.csv"

T_REG = "regla"
T_SUB = "sub-regla"


## 2. Lectura del PDF

Resolución del repositorio, ruta al PDF y texto completo; luego el cuerpo normativo anclado en `1 SAFETY`.


In [41]:
def resolver_repo() -> Path:
    env = os.environ.get("REGATAS_BASE_DIR", "").strip()
    if env:
        base = Path(env).expanduser().resolve()
        if (base / "corpus").is_dir():
            return base
        raise FileNotFoundError(f"REGATAS_BASE_DIR no contiene corpus/: {base}")
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "corpus").is_dir() and (p / "app.py").is_file():
            return p
    for p in [cwd, *cwd.parents]:
        if (p / "corpus").is_dir():
            return p
    raise FileNotFoundError(f"No se encontró corpus/ desde cwd={cwd}")


def pdf_full_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join((p.extract_text() or "") for p in reader.pages)


REPO = resolver_repo()
pdf_path = REPO / "corpus" / PDF_NAME
if not pdf_path.is_file():
    raise FileNotFoundError(pdf_path)

full = pdf_full_text(pdf_path)
m_start = re.search(r"(?m)^1[ \t]+SAFETY\b", full)
if not m_start:
    raise RuntimeError("No se encontró el ancla '1 SAFETY' (¿cambió el PDF?)")
body = full[m_start.start() :]

print("REPO:", REPO)
print("PDF:", pdf_path)
print("Caracteres (full):", len(full), "| cuerpo desde 1 SAFETY:", len(body))


REPO: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final
PDF: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/corpus/2025-2028-RRS-with-Changes-and-Corrections.pdf
Caracteres (full): 240376 | cuerpo desde 1 SAFETY: 221392


## 3. Limpieza de texto

Funciones para pie de página, colas de definiciones y ruido típico de **Part/Section + número de página** insertado por la extracción PDF.

En el armado de cada regla (`markers_to_rows`) el orden es: **prefacio `Part N` → inserciones → cabecera SECTION/Appendix → pie de página**. Así el bloque `Part 2 …` no se borra antes de poder recortar por él (si no, puede quedar pegada la cabecera SECTION siguiente, p. ej. en la regla 17).


In [42]:
def _strip_trailing_page_noise(text: str) -> str:
    t = text.rstrip()
    t = re.sub(r"(?:\n\s*)+\d{1,3}\s*$", "", t)
    return t.rstrip()


def _limpiar_cola_definicion(text: str) -> str:
    """Quita repeticiones de encabezado DEFINITIONS y números de página al pie."""
    t = re.sub(r"(?s)\s+DEFINITIONS\b.*$", "", text, flags=re.I)
    return _strip_trailing_page_noise(t.strip())


def _limpiar_texto_definicion_rule(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\n\s*DEFINITIONS\s*\n?", "\n", t)
    t = re.sub(r"\n\s*\d{1,3}\s*\n\s*", "\n", t)
    return _limpiar_cola_definicion(t)


def _limpiar_inserciones_pdf_reglas(text: str) -> str:
    """Quita encabezados repetidos de Parte/Section y número de página insertados en medio del texto.
    Debe ejecutarse después de `_recortar_prefacio_siguiente_parte` cuando el bloque `Part N …` marca el
    fin real de la regla: si se elimina antes, desaparece el ancla y queda pegada la cabecera SECTION siguiente."""
    if not text:
        return text
    t = text
    t = re.sub(
        r"(?mi)\n\s*Part\s+\d+\s+[^\n]+\s*\n(?:\s*\n)*\s*\d{1,3}\s*\n(?:\s*\n)*",
        "\n",
        t,
    )
    t = re.sub(
        r"(?mi)\n\s*SECTION\s+[A-Z]\s+[^\n]+\s*\n(?:\s*\n)*\s*\d{1,3}\s*\n(?:\s*\n)*",
        "\n",
        t,
    )
    t = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def _recortar_cabecera_siguiente_seccion(text: str) -> str:
    """Elimina al final del fragmento la cabecera SECTION/Appendix siguiente y el texto pegado hasta el fin del fragmento.
    Incluye el párrafo tipo «Section C rules do not apply…» que el PDF suele colar antes de la regla numerada siguiente."""
    t = text.rstrip()
    t = re.sub(r"(?si)\n+\s*SECTION\s+[A-Z]\b.*$", "", t)
    t = re.sub(r"(?si)\n+\s*(?:APPENDIX|Appendix)\s+[A-Z]{1,3}\b.*$", "", t)
    return t.rstrip()

def _recortar_prefacio_siguiente_parte(text: str) -> str:
    """Recorta desde PART N (cabecera de parte) y el texto que el PDF pega detrás; a veces va precedido de un nº de página suelto."""
    m = re.search(r"(?si)\n\s*(?:PART|Part)\s+\d+\b", text)
    if not m:
        return text
    t = text[: m.start()].rstrip()
    t = re.sub(r"(?si)(?:\n\s*\d{1,3}\s*)+\s*$", "", t)
    return t.rstrip()



## 4. Definiciones (DEFINITIONS → `definitions.csv`)

Extracción del bloque *DEFINITIONS* … *BASIC PRINCIPLES* en `full`. Salida: **`scripts/definitions.csv`** con columnas **id** (orden en el PDF), **termino** y **texto**.

In [43]:
def _es_termino_definicion(term: str) -> bool:
    t = term.strip()
    if len(t) < 3 or len(t) > 85:
        return False
    if not re.match(r"^[A-Z]", t):
        return False
    tl = t.lower()
    if tl.startswith("definitions"):
        return False
    if "stated below" in tl or "italic type" in tl or tl == "terminology":
        return False
    if re.match(r"^(Part|SECTION|Appendix)\b", t, re.I):
        return False
    if re.search(r"\d{4}", t) and len(t) < 25:
        return False
    return True


def extract_definitions_entries(full: str) -> list[dict[str, str]]:
    """Bloque DEFINITIONS … BASIC PRINCIPLES → filas con id, termino, texto (orden PDF)."""
    m_def = re.search(r"\bDEFINITIONS\b", full)
    m_basic = re.search(r"\n\s*BASIC PRINCIPLES\b", full)
    if not m_def or not m_basic or m_basic.start() <= m_def.start():
        return []
    block = full[m_def.start() : m_basic.start()]

    rule_lo = rule_hi = -1
    m_rule = re.search(r"(?m)^\s*Rule\s*$", block)
    if m_rule:
        rule_lo = m_rule.start()
        rule_hi = block.find("Sail the Course", rule_lo)
        if rule_hi < 0:
            rule_hi = len(block)

    rx_term = re.compile(
        r"(?m)^\s*([A-Z][A-Za-z0-9,/'’\-\; ]{1,120}?)(?:[ \t]{2,})(?=\S)"
    )
    hits = list(rx_term.finditer(block))

    ordered: list[tuple[int, str, str]] = []

    if m_rule:
        ordered.append(
            (
                rule_lo,
                "Rule",
                _limpiar_texto_definicion_rule(block[rule_lo:rule_hi]),
            )
        )

    for i, h in enumerate(hits):
        if rule_lo != -1 and rule_lo <= h.start() < rule_hi:
            continue
        raw_term = h.group(1).strip()
        term = re.sub(r" +", " ", raw_term)
        if not _es_termino_definicion(term):
            continue
        end = hits[i + 1].start() if i + 1 < len(hits) else len(block)
        if rule_lo != -1 and h.start() < rule_lo < end:
            end = rule_lo
        txt = _limpiar_cola_definicion(block[h.start() : end])
        ordered.append((h.start(), term, txt))

    ordered.sort(key=lambda x: x[0])
    return [
        {"id": str(i + 1), "termino": ter, "texto": tx}
        for i, (_, ter, tx) in enumerate(ordered)
    ]


definition_rows = extract_definitions_entries(full)
defs_path = REPO / "scripts" / DEFINITIONS_OUT
with defs_path.open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=["id", "termino", "texto"])
    w.writeheader()
    w.writerows(definition_rows)

print("Definiciones:", len(definition_rows), "→", defs_path)

Definiciones: 26 → /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/scripts/definitions.csv


## 5. Expresiones regulares y extracción de reglas

Marcadores de reglas/subreglas/apéndice sobre `body`, y armado de filas con limpieza aplicada a cada fragmento.

Incluye **índice de secciones** (`build_section_index`): cabeceras `SECTION` y `Appendix` / `APPENDIX` con título en la línea siguiente.


In [44]:
def build_section_index(body: str) -> list[tuple[int, str]]:
    """Cabeceras SECTION … / Appendix … (título en la línea siguiente). Posiciones en `body`."""
    spans: list[tuple[int, str]] = []
    for m in re.finditer(
        r"(?mi)^\s*(SECTION\s+[A-Z])\s*\n\s*([^\n]{1,120})$",
        body,
    ):
        spans.append((m.start(), f"{m.group(1).strip()} — {m.group(2).strip()}"))
    for m in re.finditer(
        r"(?mi)^\s*(?:APPENDIX|Appendix)\s+([A-Z]{1,3})\s*\n\s*([^\n]{1,120})$",
        body,
    ):
        letter = m.group(1).strip()
        title = m.group(2).strip()
        spans.append((m.start(), f"Appendix {letter} — {title}"))
    spans.sort(key=lambda x: x[0])
    return spans


def section_at(pos: int, spans: list[tuple[int, str]]) -> str:
    cur = ""
    for start, lab in spans:
        if start > pos:
            break
        cur = lab
    return cur


def collect_markers(body: str) -> list[tuple[int, int, str, str, str]]:
    m_app = re.search(r"(?m)^Appendix A\b", body)
    if m_app:
        core, appx = body[: m_app.start()], body[m_app.start() :]
    else:
        core, appx = body, ""

    rx_main = re.compile(r"(?m)^(?P<id>\d{1,2})[ \t]+(?P<title>[A-Z][^\n]{4,})$")
    rx_sub = re.compile(r"(?m)^(?P<id>\d{1,2}\.\d+(?:\.\d+)?)[ \t]+")
    rx_app = re.compile(r"(?m)^(?P<id>[A-HJ-NPR-Z]\d+\.\d+(?:\.\d+)?)[ \t]+")

    markers: list[tuple[int, int, str, str, str]] = []

    def add_from(text: str, base: int, *, allow_main: bool) -> None:
        if allow_main:
            for m in rx_main.finditer(text):
                tid, title = m.group("id"), m.group("title")
                n = int(tid)
                if n < 1 or n > 96:
                    continue
                caps = sum(1 for c in title if c.isupper())
                if len(title) > 15 and caps < len(title) * 0.45:
                    continue
                markers.append(
                    (base + m.start(), base + m.end(), tid, "main", title.strip())
                )
        for m in rx_sub.finditer(text):
            markers.append(
                (base + m.start(), base + m.end(), m.group("id"), "sub", "")
            )
        for m in rx_app.finditer(text):
            markers.append(
                (base + m.start(), base + m.end(), m.group("id"), "app", "")
            )

    add_from(core, 0, allow_main=True)
    add_from(appx, len(core), allow_main=False)
    markers.sort(key=lambda x: x[0])
    return markers


def _tipo_por_marker(kind: str) -> str:
    if kind == "sub":
        return T_SUB
    return T_REG


def _titulo_desde_primera_linea(rid: str, texto: str) -> str:
    if not texto.strip():
        return ""
    first = texto.strip().split("\n", 1)[0].strip()
    m = re.match(re.escape(rid) + r"\s+(.*)$", first)
    if m:
        return m.group(1).strip()
    return ""


def markers_to_rows(
    body: str,
    markers: list[tuple[int, int, str, str, str]],
    section_index: list[tuple[int, str]],
) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for i, (s, _e, rid, kind, titulo_main) in enumerate(markers):
        nxt = markers[i + 1][0] if i + 1 < len(markers) else len(body)
        raw = body[s:nxt].strip()
        txt = raw.strip()
        txt = _recortar_prefacio_siguiente_parte(txt)
        txt = _limpiar_inserciones_pdf_reglas(txt)
        txt = _recortar_cabecera_siguiente_seccion(txt)
        txt = _strip_trailing_page_noise(txt)
        sec = section_at(s, section_index)
        titulo = titulo_main if titulo_main else _titulo_desde_primera_linea(rid, txt)
        rows.append(
            {
                "numero_regla": rid,
                "titulo": titulo,
                "tipo": _tipo_por_marker(kind),
                "seccion": sec,
                "texto": txt,
            }
        )
    return rows


## 6. Construcción de la tabla de reglas

Arma `rule_rows` desde `body` (sin mezclar definiciones; van en `definitions.csv`).


In [45]:
section_index = build_section_index(body)
markers = collect_markers(body)
rule_rows = markers_to_rows(body, markers, section_index)
all_rows = rule_rows

print("Índice de secciones (SECTION / Appendix):", len(section_index))
print("Definiciones (definitions.csv):", len(definition_rows))
print("Reglas:", sum(1 for r in all_rows if r["tipo"] == T_REG))
print("Sub-reglas:", sum(1 for r in all_rows if r["tipo"] == T_SUB))
print("Total filas reglas:", len(all_rows))


Índice de secciones (SECTION / Appendix): 25
Definiciones (definitions.csv): 26
Reglas: 269
Sub-reglas: 200
Total filas reglas: 469


## 7. Escritura del CSV de reglas


In [46]:
out_path = REPO / "scripts" / OUT_NAME
with out_path.open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(
        f, fieldnames=["numero_regla", "titulo", "tipo", "seccion", "texto"]
    )
    w.writeheader()
    w.writerows(all_rows)

print("CSV reglas:", out_path)


CSV reglas: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/scripts/rrs_reglas_2025_2028.csv
